# Cleaning and fixing the missing data
 
### Captured at: Salvora Archipelago, Galicia (Spain)
<table>
  <tr>
    <th>Gateway</th>
    <th>Latitude</th>
    <th>Longitude</th>
    <th>Altitude</th>
  </tr>
  <tr>
    <td> 1</td>
    <td>42.46972</td>
    <td>-9.01345</td>
    <td>73</td>
  </tr>
  <tr>
    <td> 2</td>
    <td>42.49955</td>
    <td>-9.00654</td>
    <td>5</td>
  </tr>
  <tr>
    <td> 3</td>
    <td>42.50893</td>
    <td>-9.04902</td>
    <td>31</td>
  </tr>
</table>

<br>

<table>
  <tr>
    <th colspan="3"> Label and Terrain Penalty (Adjust terrain penalty with your own) </th>
  </tr>
  <tr>
    <th>Label</th>
    <th>Code</th>
    <th>Terrain Penalty</th>
  </tr>
  <tr>
  <!-- first row -->
  <tr>
    <td>Trees</td>
    <td>10</td>
    <td>0.9</td>
  </tr>
  <!-- second row -->
  <tr>
    <td>Shrubland</td>
    <td>20</td>
    <td>0.85</td>
  </tr>
  <!-- third row -->
  <tr>
    <td>Grassland</td>
    <td>30</td>
    <td>0.8</td>
  </tr>
  <!-- fourth row -->
  <tr>
    <td>Cropland</td>
    <td>40</td>
    <td>0.75</td>
  </tr>
  <!-- fifth row -->
  <tr>
    <td>Built-up</td>
    <td>50</td>
    <td>0.6</td>
  </tr>
  <!-- sixth row -->
  <tr>
    <td>Bare/ sparse vegetation</td>
    <td>60</td>
    <td>0.7</td>
  </tr>
  <!-- seventh row -->
  <tr>
    <td>Snow and ice</td>
    <td>70</td>
    <td>0.95</td>
  </tr>
  <!-- eighth row -->
  <tr>
    <td>Permanent water bodies</td>
    <td>80</td>
    <td>0.95</td>
  </tr>
  <!-- ninth row -->
  <tr>
    <td>Herbaceous wetlands</td>
    <td>90</td>
    <td>0.85</td>
  </tr>
  <!-- tenth row -->
  <tr>
    <td>Manggroves</td>
    <td>95</td>
    <td>0.9</td>
  </tr>
  <!-- eleventh row -->
  <tr>
    <td>Moss and lichen</td>
    <td>100</td>
    <td>0.95</td>
  </tr>
</table>


### Preprocessing Data 2

This notebook contains the code to preprocess the raw data 2 for the ABC2026 project. Code is similar to the preprocessing_data_1.ipynb notebook, but with different constants and columns.

In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import ee
import warnings
import os
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

True

#### 1. Initialize Google Earth Engine

In [3]:
# Run this cell only once to authenticate
ee.Authenticate()
# Try initializing Earth Engine
try:
    ee.Initialize(project=os.getenv('GEE_PROJECT_ID'))
    print("Google Earth Engine initialized")
except Exception as e:
    print("Error initializing Google Earth Engine:", e)
    exit()

Google Earth Engine initialized


#### 2. Define Constants (Adjust as Needed)

In [4]:
GATEWAYS = {
    'Gateway_1': {'lat': 42.46972, 'lon': -9.01345, 'alt': 73},
    'Gateway_2': {'lat': 42.49955, 'lon': -9.00654, 'alt': 5},
    'Gateway_3': {'lat': 42.50893, 'lon': -9.04902, 'alt': 31}
}

# Define a destination point (adjust as needed)
DESTINATION = {'lat': 42.47, 'lon': -9.01, 'alt': 0}

TX_POWER_DBM = 14

# ESA WorldCover labels
LAND_COVER_LABELS = {
    10: 'Tree',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen'
}

# Terrain penalty map (higher = less loss, lower = more loss)
PENALTY_MAP = {
    10: 0.9,   # Tree
    20: 0.85,  # Shrubland
    30: 0.8,   # Grassland
    40: 0.75,  # Cropland
    50: 0.6,   # Built-up
    60: 0.7,   # Bare/sparse
    70: 0.95,  # Ice/Snow
    80: 0.95,  # Water
    90: 0.85,  # Wetland
    95: 0.9,   # Mangroves
    100: 0.95  # Moss/lichen
}

#### 3. Define functions to preprocess data

In [5]:
def snr_to_pdr(snr, spreading_factor):
    snr_thresholds = {7: -7.5, 8: -10, 9: -12.5, 10: -15, 11: -17.5, 12: -20}
    thr = snr_thresholds.get(spreading_factor, -7.5)
    margin = snr - thr
    pdr = 1 - np.exp(-0.4 * margin)
    return np.clip(pdr, 0.0, 1.0)

def get_elevation(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    srtm = ee.Image('USGS/SRTMGL1_003')
    try:
        elev = srtm.sample(point, scale=30).first().get('elevation').getInfo()
        return float(elev) if elev is not None else np.nan
    except:
        return np.nan

def get_land_cover(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    lc = ee.ImageCollection("ESA/WorldCover/v200").first()
    try:
        code = lc.sample(point, scale=10).first().get('Map').getInfo()
        return int(code) if code is not None else np.nan
    except:
        return np.nan

def get_terrain_penalty(code):
    return PENALTY_MAP.get(code, 0.8)  # default to grassland-like

def calculate_3d_distance(lat1, lon1, alt1, lat2, lon2, alt2):
    horiz = geodesic((lat1, lon1), (lat2, lon2)).meters
    vert = abs(alt1 - alt2)
    return np.sqrt(horiz**2 + vert**2)

def calculate_path_loss(rssi, tx_power=TX_POWER_DBM):
    return tx_power - rssi

#### 4. Load and Clean the Raw Data

In [6]:
df = pd.read_csv('../raw_data/data_2.csv')
df.columns = df.columns.str.strip()  # Fix whitespace in column names

print(f"Loaded {len(df)} rows. Columns: {list(df.columns)}")

# Drop rows with missing values
df.dropna(inplace=True)
print(f"After dropping NaNs: {len(df)} rows remaining")

Loaded 1210 rows. Columns: ['#device_id', 'rssi_1', 'snr_1', 'rssi_2', 'snr_2', 'rssi_3', 'snr_3', 'spreading_factor', 'ts', 'lat', 'long', 'alt']
After dropping NaNs: 1210 rows remaining


#### 5. Calculate and Process Features

In [7]:
records = []

print(f"\nCalculating features for {len(df)} data points...")
for idx, row in df.iterrows():
    device_id = row['#device_id']
    dev_lat = row['lat']
    dev_lon = row['long']
    dev_alt = row['alt']
    ts = row['ts']
    sf = int(row['spreading_factor'])

    # Fetch elevation and land cover ONCE per device position
    elev = get_elevation(dev_lat, dev_lon)
    lc_code = get_land_cover(dev_lat, dev_lon)
    lc_label = LAND_COVER_LABELS.get(lc_code, f"Unknown ({lc_code})") if pd.notna(lc_code) else "Unknown"
    terrain_penalty = get_terrain_penalty(lc_code) if pd.notna(lc_code) else 0.8

    # Distance to destination
    dist_to_dest = calculate_3d_distance(
        dev_lat, dev_lon, dev_alt,
        DESTINATION['lat'], DESTINATION['lon'], DESTINATION['alt']
    )

    # Process each gateway
    for i in range(1, 4):
        rssi_col = f'rssi_{i}'
        snr_col = f'snr_{i}'

        rssi = row[rssi_col]
        snr = row[snr_col]

        # Skip if RSSI is 0 or NaN (not received)
        if pd.isna(rssi) or rssi == 0:
            continue

        gw = GATEWAYS[f'Gateway_{i}']
        dist_to_gw = calculate_3d_distance(
            dev_lat, dev_lon, dev_alt,
            gw['lat'], gw['lon'], gw['alt']
        )

        path_loss = calculate_path_loss(rssi)
        pdr = snr_to_pdr(snr, sf)

        records.append({
            'device_id': device_id,
            'gateway_id': f'Gateway_{i}',
            'latitude': dev_lat,
            'longitude': dev_lon,
            'altitude': dev_alt,
            'timestamp': ts,
            'RSSI': rssi,
            'SNR': snr,
            'spreading_factor': sf,
            'distance_to_start': dist_to_gw,
            'distance_to_destination': dist_to_dest,
            'elevation': elev,
            'land_cover_code': lc_code,
            'land_cover_label': lc_label,
            'terrain_penalty': terrain_penalty,
            'path_loss': path_loss,
            'PDR': pdr
        })
        
    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")


Calculating features for 1210 data points...
Processed 100 rows...
Processed 200 rows...
Processed 300 rows...
Processed 400 rows...
Processed 500 rows...
Processed 600 rows...
Processed 700 rows...
Processed 800 rows...
Processed 900 rows...
Processed 1000 rows...
Processed 1100 rows...
Processed 1200 rows...


#### 6. Finalize and Save the Processed Data

In [8]:
# Convert to DataFrame
df_out = pd.DataFrame(records)

# Drop rows where critical features failed
df_final = df_out.dropna(subset=['elevation', 'land_cover_code', 'path_loss', 'PDR'])

print("\nFinal Output Data Shape:", df_final.shape)
print("\nFinal Output Columns:", df_final.columns.tolist())
print("\nFinal Output Head:")
print(df_final.head())

print(f"Final dataset: {len(df_final)} rows")
output_filename = r'../data/processed_data2_1.csv'
df_final.to_csv(output_filename, index=False)

print(f"\nProcessed dataset saved as '{output_filename}'")


Final Output Data Shape: (2647, 17)

Final Output Columns: ['device_id', 'gateway_id', 'latitude', 'longitude', 'altitude', 'timestamp', 'RSSI', 'SNR', 'spreading_factor', 'distance_to_start', 'distance_to_destination', 'elevation', 'land_cover_code', 'land_cover_label', 'terrain_penalty', 'path_loss', 'PDR']

Final Output Head:
   device_id gateway_id   latitude  longitude   altitude     timestamp   RSSI  \
0        1.0  Gateway_1  42.470508  -9.001788  58.900002  1.575451e+09 -109.0   
1        1.0  Gateway_2  42.470508  -9.001788  58.900002  1.575451e+09 -107.0   
2        1.0  Gateway_3  42.470508  -9.001788  58.900002  1.575451e+09 -120.0   
3        1.0  Gateway_1  42.470554  -9.001778  57.000000  1.575451e+09 -109.0   
4        1.0  Gateway_2  42.470554  -9.001778  57.000000  1.575451e+09 -107.0   

    SNR  spreading_factor  distance_to_start  distance_to_destination  \
0   4.2                12         963.136866               680.232782   
1   5.2                12        32